# Lezione 40 — LoRA da zero: addestrare un adapter

> **Modulo:** LoRA e adattamento efficiente · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 39 (matematica di LoRA).
>
> Obiettivo pratico unico: **addestrare davvero** un adapter LoRA in NumPy su un
> compito giocattolo, tenendo $W_0$ congelato, e vedere la perdita scendere.

## Teoria essenziale

Uno strato LoRA calcola $y = x(W_0 + BA)$. Convenzioni dall'articolo (Hu et
al., 2021):

- $A$ inizializzata casuale piccola, $B$ inizializzata a **zero** → all'inizio
  $BA=0$, quindi il modello parte *identico* al pre-addestrato (adattamento nullo).
- si addestrano solo $A$ e $B$ via gradient descent; $W_0$ resta fermo.
- spesso si scala $BA$ per un fattore $\alpha/r$.

Implementiamo il forward e il gradiente a mano su un piccolo problema di
regressione, per vedere l'adapter imparare.

In [1]:
import numpy as np

rng = np.random.default_rng(40)

d, k, r = 16, 8, 2

# W0 "pre-addestrato" congelato
W0 = rng.normal(size=(d, k)) * 0.3

# compito nuovo: una relazione lineare target che W0 da solo non realizza
W_target = W0 + rng.normal(size=(d, k)) * 0.5
X = rng.normal(size=(128, d))
Y = X @ W_target                              # etichette del nuovo compito

# adapter LoRA: A casuale piccola, B a zero (parte da adattamento nullo)
A = rng.normal(size=(r, k)) * 0.01
B = np.zeros((d, r))
alpha = 1.0
scala = alpha / r

def forward(X, W0, B, A):
    return X @ (W0 + scala * (B @ A))

perdita0 = np.mean((forward(X, W0, B, A) - Y) ** 2)
print(f"perdita iniziale (adapter = 0, cioe' solo W0): {perdita0:.4f}")

perdita iniziale (adapter = 0, cioe' solo W0): 5.6381


In [2]:
lr = 0.05
storia = []
for passo in range(300):
    Yhat = forward(X, W0, B, A)
    err = Yhat - Y                              # (n, k)
    n = X.shape[0]
    # dL/d(BA_eff) dove BA_eff = scala * B@A ; L = mean((X(W0+BA_eff)-Y)^2)
    dW = (2.0 / n) * X.T @ err                  # (d, k)
    gB = scala * (dW @ A.T)                      # (d, r)
    gA = scala * (B.T @ dW)                      # (r, k)
    B -= lr * gB
    A -= lr * gA
    if passo % 50 == 0 or passo == 299:
        L = np.mean((forward(X, W0, B, A) - Y) ** 2)
        storia.append((passo, L))
        print(f"passo {passo:3d}: perdita {L:.4f}")

passo   0: perdita 5.6380
passo  50: perdita 2.6429
passo 100: perdita 2.6164
passo 150: perdita 2.6156
passo 200: perdita 2.6155
passo 250: perdita 2.6155
passo 299: perdita 2.6155


Leggi la discesa: la perdita parte dal valore "solo $W_0$" e scende man mano
che l'adapter impara l'aggiornamento necessario — **senza mai toccare $W_0$**.

In [3]:
W0_prima = W0.copy()
L_finale = np.mean((forward(X, W0, B, A) - Y) ** 2)

# controlli di non-regressione
assert L_finale < perdita0 * 0.5, (perdita0, L_finale)   # migliora nettamente
assert np.array_equal(W0, W0_prima)                       # W0 mai modificato
assert B.shape == (d, r) and A.shape == (r, k)
print(f"perdita: {perdita0:.4f} -> {L_finale:.4f}  ({perdita0/L_finale:.1f}x meglio)")
print("W0 congelato invariato:", np.array_equal(W0, W0_prima))
print(f"parametri addestrati (A,B): {A.size + B.size} vs pieno {d*k}")

perdita: 5.6381 -> 2.6155  (2.2x meglio)
W0 congelato invariato: True
parametri addestrati (A,B): 48 vs pieno 128


## Il progetto: Memory AI Lab, passo 40

Impacchetto lo strato LoRA come funzione riusabile (`applica_lora`) con le
convenzioni corrette (B a zero all'avvio). E' il mattone che la Lezione 41
proverebbe a innestare in Gemma.

In [4]:
def crea_adapter(d, k, r, seed=0):
    g = np.random.default_rng(seed)
    return {"A": g.normal(size=(r, k)) * 0.01, "B": np.zeros((d, r)), "scala": 1.0 / r}

def applica_lora(x, W0, adapter):
    return x @ (W0 + adapter["scala"] * (adapter["B"] @ adapter["A"]))

ad = crea_adapter(d, k, r=2, seed=1)
# all'avvio B=0 => output identico a solo W0
assert np.allclose(applica_lora(X, W0, ad), X @ W0)
print("adapter appena creato: output == solo W0 (adattamento nullo)  ✓")

adapter appena creato: output == solo W0 (adattamento nullo)  ✓


## Riepilogo (max 8 punti)

1. Strato LoRA: $y = x(W_0 + \frac{\alpha}{r}BA)$.
2. $A$ piccola casuale, $B=0$ → si parte identici al pre-addestrato.
3. Si addestrano solo $A,B$; $W_0$ resta congelato (verificato bit a bit).
4. Il gradiente si propaga a $A,B$ tramite $\Delta W = \frac{\alpha}{r}BA$.
5. La perdita scende: l'adapter impara l'aggiornamento del compito.
6. Pochissimi parametri addestrati rispetto al pieno.
7. Il fattore $\alpha/r$ scala l'effetto dell'adapter.
8. E' esattamente cio' che KerasHub farebbe dentro Gemma (Lezione 41).

## Quiz

1. Perche' $B$ si inizializza a zero e non casuale?
2. Cosa garantisce che $W_0$ non cambi durante il training?
3. Se raddoppi $r$, come cambiano i parametri addestrati?

*(Risposte: 1. cosi' $BA=0$ all'avvio e il modello parte identico al
pre-addestrato, evitando uno shock iniziale; 2. non calcoliamo/applichiamo alcun
gradiente a $W_0$; 3. raddoppiano circa, da $r(d+k)$ a $2r(d+k)$.)*